# Complete the five-fold CNN evaluation
The fold-5 checkpoint and test report already exist. This notebook trains and evaluates folds 1–4 with the same fixed settings, then combines all five test reports. Select a Colab GPU runtime and run the cells in order. Do not change the settings based on the fold-5 test result.

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive

project_dir = Path('/content/soundview')
data_dir = Path('/content/datasets/ESC-50')

if project_dir.exists():
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AndreuLopezz/12607559_SoundView-environmental-sound-recognition.git', str(project_dir)], check=True)
if not data_dir.exists():
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/karolpiczak/ESC-50.git', str(data_dir)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project_dir / 'requirements.txt')], check=True)
sys.path.insert(0, str(project_dir))
drive.mount('/content/drive')

## Fixed experiment settings
Only change the Drive paths if your files are in different folders. Folds 1–4 use the settings chosen before the fold-5 test was measured.

In [ ]:
EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 0.001
PATIENCE = 5
SEED = 42
RUNS_DIR = Path('/content/drive/MyDrive/SoundView/cnn_fivefold')
FOLD5_REPORT = Path('/content/drive/MyDrive/SoundView/cnn_fold5_40epochs/test_evaluation.json')

In [ ]:
import json

from soundview.evaluate_cnn import evaluate_cnn
from soundview.train_cnn import train_cnn

expected_settings = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'patience': PATIENCE,
    'seed': SEED,
}

fold5_training = json.loads((FOLD5_REPORT.parent / 'training_summary.json').read_text(encoding='utf-8'))
fold5_result = json.loads(FOLD5_REPORT.read_text(encoding='utf-8'))
if fold5_training['settings'] != expected_settings or fold5_training['test_fold_reserved'] != 5:
    raise ValueError('The saved fold-5 run used different settings')
if fold5_result['test_fold'] != 5 or fold5_result['checkpoint_epoch'] != fold5_training['best_epoch']:
    raise ValueError('The fold-5 test report does not match its training run')

for fold in range(1, 5):
    fold_dir = RUNS_DIR / f'fold{fold}'
    checkpoint = fold_dir / 'best_cnn.pt'
    training_summary = fold_dir / 'training_summary.json'
    test_report = fold_dir / 'test_evaluation.json'

    if test_report.exists() and not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint for completed fold {fold}')
    # Check saved settings before reusing a fold after a disconnected session.
    if checkpoint.exists() or test_report.exists():
        if not training_summary.exists():
            raise FileNotFoundError(f'Missing training summary for fold {fold}')
        previous = json.loads(training_summary.read_text(encoding='utf-8'))
        if previous['settings'] != expected_settings or previous['test_fold_reserved'] != fold:
            raise ValueError(f'Saved fold {fold} used different settings')
        if test_report.exists():
            previous_test = json.loads(test_report.read_text(encoding='utf-8'))
            if previous_test['test_fold'] != fold or previous_test['checkpoint_epoch'] != previous['best_epoch']:
                raise ValueError(f'Saved test report does not match fold {fold}')

    if not checkpoint.exists():
        print(f'Training fold {fold}...')
        train_cnn(
            data_dir=data_dir,
            output_dir=fold_dir,
            test_fold=fold,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            patience=PATIENCE,
            seed=SEED,
        )

    if not test_report.exists():
        print(f'Evaluating test fold {fold}...')
        evaluate_cnn(data_dir, checkpoint, test_report, test_fold=fold, batch_size=BATCH_SIZE)
    else:
        print(f'Fold {fold} already evaluated; using its saved report.')

In [ ]:
from soundview.fivefold import combine_fold_reports

report_paths = [RUNS_DIR / f'fold{fold}' / 'test_evaluation.json' for fold in range(1, 5)]
report_paths.append(FOLD5_REPORT)
summary_path = RUNS_DIR / 'fivefold_summary.json'
summary = combine_fold_reports(report_paths, summary_path)

for fold in summary['folds']:
    print(f"Fold {fold['test_fold']}: accuracy {fold['accuracy']:.4f}, macro-F1 {fold['macro_f1']:.4f}")
print(f"Mean fold macro-F1: {summary['mean_fold_macro_f1']:.4f}")
print(f"Pooled macro-F1 across 2,000 clips: {summary['pooled']['macro_f1']:.4f}")
print(f'Report saved to {summary_path}')